# 04. Unsupervised Analysis

This notebook performs unsupervised clustering on the 20,013-dimensional feature space using `MiniBatchKMeans` with `n_clusters=2`.
We train it without labels and then evaluate its correspondence to the true labels (Correct vs Incorrect option) using:
- **Silhouette Score** (measuring cluster tightness and separation)
- **Cluster Purity** (post-hoc alignment with the true binary labels)


In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.sparse import load_npz

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from collections import Counter

PROCESSED_DIR = Path("../data/processed")

print("Loading validation feature matrix and labels for clustering analysis...")
# We use validation set to avoid massive memory usage on silhouette score
X_val = load_npz(PROCESSED_DIR / "X_val_features.npz")
y_val = np.load(PROCESSED_DIR / "y_val.npy")

print(f"Features: {X_val.shape}")
print(f"Labels: {y_val.shape}")


Loading validation feature matrix and labels for clustering analysis...
Features: (35436, 20013)
Labels: (35436,)


In [2]:
print("Running MiniBatchKMeans (n_clusters=2)...")
kmeans = MiniBatchKMeans(n_clusters=2, random_state=42, batch_size=1024)
clusters = kmeans.fit_predict(X_val)

def cluster_purity(y_true, y_pred):
    """Calculate cluster purity: how well do the found clusters map to the true classes?"""
    total_correct = 0
    for cluster_id in np.unique(y_pred):
        # Get true labels for items in this cluster
        cluster_labels = y_true[y_pred == cluster_id]
        if len(cluster_labels) == 0:
            continue
        # Find the most common true label in this cluster
        most_common = Counter(cluster_labels).most_common(1)[0][1]
        total_correct += most_common
    return total_correct / len(y_true)

purity = cluster_purity(y_val, clusters)
print(f"Cluster Purity: {purity:.4f}")

# Subsample for silhouette score since it's O(N^2)
print("Calculating Silhouette Score (on a random subsample of 10,000 points)...")
np.random.seed(42)
indices = np.random.choice(X_val.shape[0], size=min(10000, X_val.shape[0]), replace=False)
# Convert to CSR format because COO matrix doesn't support indexing
X_val = X_val.tocsr()
X_sub = X_val[indices]
clusters_sub = clusters[indices]

sil_score = silhouette_score(X_sub, clusters_sub)
print(f"Silhouette Score: {sil_score:.4f}")


Running MiniBatchKMeans (n_clusters=2)...
Cluster Purity: 0.7500
Calculating Silhouette Score (on a random subsample of 10,000 points)...
Silhouette Score: 0.5448
